In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [2]:
df = pd.read_excel("default of credit card clients.xls", header=1)
df = df.drop(columns=df.columns[0])

x = df.drop(columns=['default payment next month']).values
y = df['default payment next month'].values

scaler = StandardScaler()
x = scaler.fit_transform(x)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
x_test = torch.tensor(x_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)


In [3]:
class CreditDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
        
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

batch_size = 64

train_dataset = CreditDataset(x_train, y_train)
test_dataset = CreditDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [4]:
class CreditRiskNN(nn.Module):
    def __init__(self, input_dim):
        super(CreditRiskNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        return self.model(x)

input_dim = x_train.shape[1]
model = CreditRiskNN(input_dim)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


CreditRiskNN(
  (model): Sequential(
    (0): Linear(in_features=23, out_features=64, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=32, out_features=1, bias=True)
    (9): Sigmoid()
  )
)

In [5]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)


In [6]:
# Cell 6: Training loop
epochs = 50
best_loss = np.inf

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * x_batch.size(0)
    
    train_loss /= len(train_loader.dataset)
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * x_batch.size(0)
    val_loss /= len(test_loader.dataset)
    
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
    
    print(f"Epoch [{epoch+1}/{epochs}]  Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}")

Epoch [1/50]  Train Loss: 0.5473  Val Loss: 0.4477
Epoch [2/50]  Train Loss: 0.4621  Val Loss: 0.4427
Epoch [3/50]  Train Loss: 0.4507  Val Loss: 0.4410
Epoch [4/50]  Train Loss: 0.4462  Val Loss: 0.4378
Epoch [5/50]  Train Loss: 0.4440  Val Loss: 0.4371
Epoch [6/50]  Train Loss: 0.4402  Val Loss: 0.4362
Epoch [7/50]  Train Loss: 0.4395  Val Loss: 0.4368
Epoch [8/50]  Train Loss: 0.4383  Val Loss: 0.4358
Epoch [9/50]  Train Loss: 0.4373  Val Loss: 0.4358
Epoch [10/50]  Train Loss: 0.4365  Val Loss: 0.4359
Epoch [11/50]  Train Loss: 0.4368  Val Loss: 0.4352
Epoch [12/50]  Train Loss: 0.4348  Val Loss: 0.4347
Epoch [13/50]  Train Loss: 0.4333  Val Loss: 0.4359
Epoch [14/50]  Train Loss: 0.4334  Val Loss: 0.4343
Epoch [15/50]  Train Loss: 0.4337  Val Loss: 0.4349
Epoch [16/50]  Train Loss: 0.4323  Val Loss: 0.4334
Epoch [17/50]  Train Loss: 0.4321  Val Loss: 0.4349
Epoch [18/50]  Train Loss: 0.4325  Val Loss: 0.4329
Epoch [19/50]  Train Loss: 0.4343  Val Loss: 0.4335
Epoch [20/50]  Train 

In [7]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

y_pred = []
with torch.no_grad():
    for x_batch, _ in test_loader:
        x_batch = x_batch.to(device)
        outputs = model(x_batch)
        y_pred.extend((outputs.cpu().numpy() > 0.5).astype(int))

y_pred = np.array(y_pred).flatten()

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))


C:\Users\Win10\AppData\Local\Temp\ipykernel_17120\845407743.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model.pth"))


Accuracy: 0.8183333333333334
Precision: 0.6781954887218045
Recall: 0.3398643556895252
F1 Score: 0.4528112449799197


In [8]:
# Example: one new customer
new_customer = np.array([[
    200000,  # LIMIT_BAL
    2,       # SEX (2 = female)
    2,       # EDUCATION (2 = university)
    1,       # MARRIAGE (1 = married)
    35,      # AGE
    0, 0, 0, 0, 0, 0,   # PAY_0 to PAY_6
    5000, 4500, 4000, 3500, 3000, 2500,  # BILL_AMT1–6
    2000, 2000, 1500, 1500, 1000, 1000   # PAY_AMT1–6
]])


In [9]:
new_customer_scaled = scaler.transform(new_customer)


In [10]:
new_customer_tensor = torch.tensor(
    new_customer_scaled, dtype=torch.float32
).to(device)


In [11]:
model.eval()
with torch.no_grad():
    prob = model(new_customer_tensor).item()


In [12]:
print(f"Probability of default: {prob:.4f}")

if prob > 0.5:
    print("⚠️ High Credit Risk (Likely to Default)")
else:
    print("✅ Low Credit Risk")


Probability of default: 0.0691
✅ Low Credit Risk


In [13]:
import joblib

joblib.dump(scaler, "scaler.pkl")
print("Scaler saved successfully")


Scaler saved successfully
